<a href="https://colab.research.google.com/github/efthymios19/HYDRUS_GoogleColab/blob/main/Richardsequation/Richards%20Equation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Simple 1D Richards' Equation Model

This is a 1D Richards' equation model, written by Andrew Ireson, 3 November 2015.

Το υλικό τροποποιήθηκε με βάση τις απαιτήσεις του διδασκόμενου υλικού

In [2]:
!rm -r -- HYDRUS_GoogleColab/
!ls

In [3]:
# Clone github repository to import SWRC,C,K module

!git clone https://github.com/efthymios19/HYDRUS_GoogleColab.git

In [4]:
!ls HYDRUS_GoogleColab/

In [5]:
# Import all of the basic libraries (you will always need these)
%matplotlib inline
from matplotlib import pyplot as pl
import numpy as np

# Import a library that contains soil moisture properties and functions
import HYDRUS_GoogleColab.Richardsequation.vanGenuchten as vg

# Import ODE solvers
from scipy.interpolate import interp1d
from scipy.integrate import odeint

# Select which soil properties to use
#p=vg.HygieneSandstone()
#p=vg.GuelphLoamWetting()
p=vg.GuelphLoamDrying()
#p=vg.BeitNetofaClay()

## Mέθοδος των γραμμών

Η συνάρτηση `RichardsModel` υπολογίζει μία τιμή $d\psi_i/dt$ ανά κελί από τη διαφορά των ροών στις δύο έδρες του. Η αγωγιμότητα σε κάθε εσωτερική έδρα είναι ο αριθμητικός μέσος των αγωγιμοτήτων στα γειτονικά κέντρα. Το `odeint` ολοκληρώνει αυτές τις εξισώσεις στον χρόνο.


In [16]:
# Richards equation solver
# This is a function that calculated the right hand side of Richards' equation. You
# will not need to modify this function, unless you are doing something advanced. 
# This block of code must be executed so that the function can be later called.

def RichardsModel(psi,t,dz,n,p,vg,qTop,qBot,psiTop,psiBot):
       
    # Basic properties:
    C=vg.CFun(psi,p)
   
    # initialize vectors:
    q=np.zeros(n+1)
    
    # Upper boundary
    if qTop == []: # (Dirichlet) 
        KTop=vg.KFun(np.zeros(1)+psiTop,p).item()
        q[n]=-KTop*((psiTop-psi[n-1])/dz*2+1)
    else:
        q[n]=qTop #(Neumann) prescribe flux
    
    # Lower boundary
    if qBot == []:
        if psiBot == []:
            # Free drainage
            KBot=vg.KFun(np.zeros(1)+psi[0],p)
            q[0]=-KBot.item()
        else:
            # Type 1 boundary (fixed pressure–Dirichlet)
            KBot=vg.KFun(np.zeros(1)+psiBot,p).item()
            q[0]=-KBot*((psi[0]-psiBot)/dz*2+1.0)    
    else:
        # Type 2 Neumann
        q[0]=qBot
    
    # Internal nodes
    i=np.arange(0,n-1)
    Knodes=vg.KFun(psi,p)
    Kmid=(Knodes[i+1]+Knodes[i])/2.0
    
    j=np.arange(1,n)
    q[j]=-Kmid*((psi[i+1]-psi[i])/dz+1.0)
    
    
    
    # Continuity
    i=np.arange(0,n)
    dpsidt=(-(q[i+1]-q[i])/dz)/C
    
    return dpsidt

In [17]:
psi = np.linspace(-10,0)
theta = vg.thetaFun(psi,p)
C=vg.CFun(psi,p)
K=vg.KFun(psi,p)

pl.rcParams['figure.figsize'] = (5.0, 10.0)
pl.subplot(311)
pl.plot(psi,theta)
pl.ylabel(r'$\theta$', fontsize=20)
pl.subplot(312)
pl.plot(psi,C)
pl.ylabel(r'$C$',fontsize=20)
pl.subplot(313)
pl.plot(psi,K)
pl.ylabel(r'$K$', fontsize=20)
pl.xlabel(r'$\psi$', fontsize=20)


## Τι ορίζουμε πριν τρέξουμε το μοντέλο;

### Συνοριακές συνθήκες

Συμπληρώστε αριθμό για την επιλεγμένη συνθήκη και χρησιμοποιήστε `[]` για την αντίστοιχη μεταβλητή που μένει ανενεργή. Αν οριστεί ροή, ο κώδικας τη χρησιμοποιεί κατά προτεραιότητα.

| Άκρο | Συνθήκη | Ρύθμιση | Φυσική ερμηνεία |
|---|---|---|---|
| Άνω | Επιβαλλόμενη ροή (Neumann) | Αριθμητικό `qTop`, `psiTop=[]` | Αρνητική τιμή σημαίνει διήθηση. Η τιμή 0 σημαίνει μηδενική ροή. |
| Άνω | Σταθερό φορτίο πίεσης (Dirichlet) | `qTop=[]`, αριθμητικό `psiTop` | Για παράδειγμα, 0 m για νερό σε ατμοσφαιρική πίεση στην επιφάνεια. |
| Κάτω | Επιβαλλόμενη ροή (Neumann) | Αριθμητικό `qBot`, `psiBot=[]` | Αρνητική τιμή σημαίνει εκροή προς τα κάτω. Η τιμή 0 δίνει αδιαπέρατη βάση. |
| Κάτω | Σταθερό φορτίο πίεσης (Dirichlet) | `qBot=[]`, αριθμητικό `psiBot` | Για παράδειγμα, 0 m για στάθμη νερού στη βάση. |
| Κάτω | Ελεύθερη στράγγιση | `qBot=[]`, `psiBot=[]` | Μηδενική κλίση φορτίου πίεσης στη βάση. Η βαρύτητα δίνει $q=-K(\psi_0)$. |

Στην κορυφή χρειάζεται αριθμητική τιμή για μία από τις δύο συνθήκες. Με σταθερό φορτίο, η απόσταση από το όριο έως το πλησιέστερο κέντρο είναι $\Delta z/2$, εξ ου ο παράγοντας 2 στον υπολογισμό της κλίσης.

**Τρέχουσα ρύθμιση.** Στην κορυφή επιβάλλεται `qTop=-0.01` m/day, δηλαδή διήθηση 10 mm/day. Στη βάση εφαρμόζεται ελεύθερη στράγγιση.

### Χωρική και χρονική διακριτοποίηση

Η στήλη των 10 m χωρίζεται σε 100 κελιά πάχους 0.1 m. Τα κέντρα τους είναι στα 0.05, 0.15, …, 9.95 m από τη βάση. Ο κώδικας αποθηκεύει 100 τιμές $\psi$ στα κέντρα και υπολογίζει 101 ροές στις έδρες. Οι 101 χρόνοι από 0 έως 10 ημέρες, ανά 0.1 ημέρα, είναι χρόνοι αποθήκευσης αποτελεσμάτων. Ο επιλυτής επιλέγει τα εσωτερικά του χρονικά βήματα.

### Αρχική κατάσταση

Το `psi0=-z` δίνει $\psi=-0.05$ m στο χαμηλότερο κέντρο και $\psi=-9.95$ m στο ανώτερο. Στην αρχή $H=\psi+z=0$ σε όλη τη στήλη. Πρόκειται για υδροστατικό αρχικό προφίλ που αντιστοιχεί σε μηδενικό φορτίο πίεσης στη βάση. Οι εσωτερικές ροές είναι αρχικά μηδενικές. Η επιβαλλόμενη διήθηση και η ελεύθερη στράγγιση δημιουργούν ροές στα όρια μόλις ξεκινήσει η προσομοίωση. Η αρχική ογκομετρική υγρασία προκύπτει από τη συνάρτηση συγκράτησης $\theta(\psi)$ του επιλεγμένου εδάφους.


In [ ]:
# This block of code sets up and runs the model

# Boundary conditions
qTop= [] #m/day 10 mm/day infiltration if sine is minus
qBot= []
psiTop= 0
psiBot= 0

# Grid in space
dz=0.1 #m
ProfileDepth=10 #m
z=np.arange(dz/2.0,ProfileDepth,dz)
n=z.size

# Grid in time
t = np.linspace(0,10,101) #10 days per 0.1 day

# Initial conditions
psi0=-z

# Solve
psi=odeint(RichardsModel,psi0,t,args=(dz,n,p,vg,qTop,qBot,psiTop,psiBot),mxstep=5000000)

print ("Model run successfully")

1. Εκτελέστε τη βασική περίπτωση. Εντοπίστε την κορυφή και τη βάση στα προφίλ. Εξηγήστε γιατί στο διάγραμμα εμφανίζεται το `z-ProfileDepth`, με μηδέν στην επιφάνεια.
2. Κρατήστε τη διήθηση ίδια και θέστε `qBot=0.0`. Συγκρίνετε την αποθήκευση νερού και την εκροή στη βάση με τη βασική περίπτωση.
3. Επαναφέρετε τη στράγγιση, θέστε `qTop=0.0` και συγκρίνετε την εξέλιξη της υγρασίας. Προβλέψτε πρώτα το πρόσημο της μεταβολής αποθήκευσης.
4. Επαναφέρετε τις αρχικές ρυθμίσεις και υποδιπλασιάστε το `dz`. Συγκρίνετε τα προφίλ στον ίδιο χρόνο. Πόσα κελιά και πόσες έδρες έχει τώρα η στήλη;



In [19]:
# Post process model output to get useful information

# Get water content
theta=vg.thetaFun(psi,p)

# Get total profile storage
S=theta.sum(axis=1)*dz

# Get change in storage [dVol]
dS=np.zeros(S.size)
dS[1:]=np.diff(S)/(t[1]-t[0])

# Get infiltration flux
if qTop == []:
    KTop=vg.KFun(np.zeros(1)+psiTop,p)
    qI=-KTop*((psiTop-psi[:,n-1])/dz*2+1)
else:
    qI=np.zeros(t.size)+qTop
    
# Get rescharge flux
if qBot == []:
    if psiBot == []:
        # Free drainage
        KBot=vg.KFun(psi[:,0],p)
        qD=-KBot
    else:
        # Type 1 boundary
        KBot=vg.KFun(np.zeros(1)+psiBot,p)
        qD=-KBot*((psi[:,0]-psiBot)/dz*2+1.0)
else:
    qD=np.zeros(t.size)+qBot
    

In [20]:
# Plot vertical profiles
pl.rcParams['figure.figsize'] = (10.0, 10.0)
for i in range(0,t.size-1,10):
    pl.subplot(121)
    pl.plot(psi[i,:],z-ProfileDepth, label=f"{t[i]+1:g} days")
    pl.subplot(122)
    pl.plot(theta[i,:],z-ProfileDepth)

pl.subplot(121)
pl.ylabel('Elevation [m]',fontsize=20)
pl.xlabel(r'$\psi$ [m]',fontsize=20)
pl.legend()
pl.subplot(122)
pl.xlabel(r'$\theta$ [-]',fontsize=20)

In [83]:
# Plot timeseries
dt = t[2]-t[1]
pl.plot(t,dS,label='Change in storage')
pl.plot(t,-qI,label='Infiltration')
pl.plot(t,-qD,label='Recharge')
pl.legend(loc="upper right")
pl.xlabel('Days')
pl.ylabel('m')
pl.ylim((0,2))
#pl.tight_layout()